In [ ]:
pip install pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.1 MB/s eta 0:00:00


In [ ]:
"""
ARIMAX Model Exploration for Sri Lankan Tourism Prediction
============================================================
Author: ML Research Team
Date: December 2025
Description: Production-ready ARIMAX implementation with proper time-series
             validation, hyperparameter tuning, and comprehensive evaluation.
"""

import os
import logging
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Tuple, Dict, List

# Statistical and ML Libraries
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import itertools
import json
import pickle

warnings.filterwarnings('ignore')

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

def setup_logging(log_dir: str = 'logs') -> logging.Logger:
    """
    Setup comprehensive logging for model exploration.

    Args:
        log_dir: Directory to save log files

    Returns:
        Logger instance
    """
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = os.path.join(log_dir, f'arimax_exploration_{timestamp}.log')

    # Configure logger
    logger = logging.getLogger('ARIMAX_Explorer')
    logger.setLevel(logging.INFO)

    # File handler
    fh = logging.FileHandler(log_file)
    fh.setLevel(logging.INFO)

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)

    # Formatter
    formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)

    logger.addHandler(fh)
    logger.addHandler(ch)

    logger.info("="*80)
    logger.info("ARIMAX MODEL EXPLORATION INITIATED")
    logger.info("="*80)

    return logger


# ============================================================================
# DATA LOADING AND PREPROCESSING
# ============================================================================

class ARIMAXDataProcessor:
    """
    ARIMAX-specific data processing including feature engineering and scaling.
    """

    def __init__(self, logger: logging.Logger):
        self.logger = logger
        self.exog_scaler = StandardScaler()
        self.feature_columns = None
        self.target_column = 'arrivals'

    def load_data(self, file_path: str) -> pd.DataFrame:
        """Load and validate the preprocessed dataset."""
        self.logger.info(f"Loading data from: {file_path}")

        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)

        self.logger.info(f"Data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")
        self.logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")

        return df

    def check_stationarity(self, series: pd.Series, name: str = "Series") -> bool:
        """
        Check stationarity using Augmented Dickey-Fuller test.

        Args:
            series: Time series to test
            name: Name of the series for logging

        Returns:
            True if stationary, False otherwise
        """
        result = adfuller(series.dropna())
        self.logger.info(f"\nStationarity Test - {name}:")
        self.logger.info(f"  ADF Statistic: {result[0]:.6f}")
        self.logger.info(f"  p-value: {result[1]:.6f}")
        self.logger.info(f"  Critical Values:")
        for key, value in result[4].items():
            self.logger.info(f"    {key}: {value:.3f}")

        is_stationary = result[1] < 0.05
        self.logger.info(f"  Stationary: {is_stationary}")

        return is_stationary

    def prepare_features(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
        """
        Prepare features for ARIMAX model.

        Args:
            df: Input dataframe

        Returns:
            Tuple of (processed_df, exogenous_feature_columns)
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("FEATURE PREPARATION FOR ARIMAX")
        self.logger.info("="*80)

        # Identify exogenous features (all except date, arrivals, and derived columns)
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        exog_features = [col for col in df.columns if col not in exclude_cols]

        self.logger.info(f"\nExogenous features identified: {len(exog_features)}")
        self.logger.info(f"Features: {exog_features}")

        # Create lagged features for key exogenous variables
        lag_features = ['usd_lkr', 'web_search', 'temperature', 'event_encoded']
        self.logger.info(f"\nCreating lagged features for: {lag_features}")

        for feature in lag_features:
            if feature in df.columns:
                df[f'{feature}_lag1'] = df[feature].shift(1)
                df[f'{feature}_lag7'] = df[feature].shift(7)
                exog_features.extend([f'{feature}_lag1', f'{feature}_lag7'])

        # Create rolling statistics
        self.logger.info("\nCreating rolling statistics features...")
        window_sizes = [7, 14, 30]
        rolling_features = ['temperature', 'humidity', 'web_search']

        for feature in rolling_features:
            if feature in df.columns:
                for window in window_sizes:
                    df[f'{feature}_rolling_mean_{window}'] = df[feature].rolling(window=window, min_periods=1).mean()
                    df[f'{feature}_rolling_std_{window}'] = df[feature].rolling(window=window, min_periods=1).std()
                    exog_features.extend([
                        f'{feature}_rolling_mean_{window}',
                        f'{feature}_rolling_std_{window}'
                    ])

        # Add temporal features
        self.logger.info("\nCreating temporal features...")
        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['day_of_year'] = df['date'].dt.dayofyear

        # Cyclical encoding for temporal features
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

        temporal_features = ['day_of_week', 'day_of_month', 'month', 'quarter',
                            'day_of_year', 'month_sin', 'month_cos', 'day_sin', 'day_cos']
        exog_features.extend(temporal_features)

        # Handle missing values
        self.logger.info("\nHandling missing values...")
        initial_missing = df[exog_features].isnull().sum().sum()
        df[exog_features] = df[exog_features].fillna(method='ffill').fillna(method='bfill')
        final_missing = df[exog_features].isnull().sum().sum()
        self.logger.info(f"Missing values before: {initial_missing}, after: {final_missing}")

        self.feature_columns = exog_features
        self.logger.info(f"\nTotal exogenous features: {len(exog_features)}")

        return df, exog_features

    def scale_exogenous_features(self, train_exog: np.ndarray,
                                val_exog: np.ndarray,
                                test_exog: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Scale exogenous variables for numerical stability.

        Args:
            train_exog: Training exogenous features
            val_exog: Validation exogenous features
            test_exog: Test exogenous features

        Returns:
            Tuple of scaled arrays
        """
        self.logger.info("\nScaling exogenous features for numerical stability...")

        # Fit scaler on training data only
        train_scaled = self.exog_scaler.fit_transform(train_exog)
        val_scaled = self.exog_scaler.transform(val_exog)
        test_scaled = self.exog_scaler.transform(test_exog)

        self.logger.info(f"Scaling completed - Mean: {train_scaled.mean():.4f}, Std: {train_scaled.std():.4f}")

        return train_scaled, val_scaled, test_scaled


# ============================================================================
# DATA SPLITTING
# ============================================================================

def time_series_split(df: pd.DataFrame,
                     train_ratio: float = 0.75,
                     val_ratio: float = 0.15,
                     logger: logging.Logger = None) -> Tuple:
    """
    Time-series aware data splitting (75/15/15).

    Args:
        df: Input dataframe
        train_ratio: Proportion for training
        val_ratio: Proportion for validation
        logger: Logger instance

    Returns:
        Tuple of (train_df, val_df, test_df, train_idx, val_idx, test_idx)
    """
    logger.info("\n" + "="*80)
    logger.info("TIME-SERIES AWARE DATA SPLITTING")
    logger.info("="*80)

    n = len(df)
    train_size = int(n * train_ratio)
    val_size = int(n * val_ratio)

    train_df = df.iloc[:train_size].copy()
    val_df = df.iloc[train_size:train_size + val_size].copy()
    test_df = df.iloc[train_size + val_size:].copy()

    logger.info(f"\nTotal samples: {n}")
    logger.info(f"Train: {len(train_df)} samples ({train_ratio*100:.1f}%) - {train_df['date'].min()} to {train_df['date'].max()}")
    logger.info(f"Validation: {len(val_df)} samples ({val_ratio*100:.1f}%) - {val_df['date'].min()} to {val_df['date'].max()}")
    logger.info(f"Test: {len(test_df)} samples ({(1-train_ratio-val_ratio)*100:.1f}%) - {test_df['date'].min()} to {test_df['date'].max()}")

    return (train_df, val_df, test_df,
            range(len(train_df)),
            range(len(train_df), len(train_df) + len(val_df)),
            range(len(train_df) + len(val_df), n))


# ============================================================================
# HYPERPARAMETER TUNING
# ============================================================================

class ARIMAXTuner:
    """
    Hyperparameter tuning for ARIMAX models using grid search.
    """

    def __init__(self, logger: logging.Logger):
        self.logger = logger
        self.best_params = None
        self.best_model = None
        self.best_aic = np.inf
        self.tuning_results = []

    def generate_param_combinations(self,
                                    p_range: range = range(0, 3),
                                    d_range: range = range(0, 2),
                                    q_range: range = range(0, 3)) -> List[Tuple]:
        """
        Generate parameter combinations for grid search.

        Args:
            p_range: Range for AR order
            d_range: Range for differencing order
            q_range: Range for MA order

        Returns:
            List of parameter tuples
        """
        param_combinations = list(itertools.product(p_range, d_range, q_range))
        self.logger.info(f"\nGenerated {len(param_combinations)} parameter combinations")
        return param_combinations

    def tune_hyperparameters(self,
                           train_endog: pd.Series,
                           train_exog: np.ndarray,
                           val_endog: pd.Series,
                           val_exog: np.ndarray,
                           param_combinations: List[Tuple]) -> Dict:
        """
        Perform grid search for hyperparameter tuning.

        Args:
            train_endog: Training target variable
            train_exog: Training exogenous features
            val_endog: Validation target variable
            val_exog: Validation exogenous features
            param_combinations: List of (p, d, q) tuples to test

        Returns:
            Dictionary with best parameters
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("HYPERPARAMETER TUNING - GRID SEARCH")
        self.logger.info("="*80)

        self.logger.info(f"\nTesting {len(param_combinations)} parameter combinations...")

        for idx, (p, d, q) in enumerate(param_combinations, 1):
            try:
                # Fit model on training data
                model = SARIMAX(train_endog,
                              exog=train_exog,
                              order=(p, d, q),
                              enforce_stationarity=False,
                              enforce_invertibility=False)

                fitted_model = model.fit(disp=False, maxiter=200)

                # Evaluate on validation set
                val_predictions = fitted_model.forecast(steps=len(val_endog), exog=val_exog)
                val_mse = mean_squared_error(val_endog, val_predictions)
                val_rmse = np.sqrt(val_mse)

                # Record results
                result = {
                    'order': (p, d, q),
                    'aic': fitted_model.aic,
                    'bic': fitted_model.bic,
                    'val_mse': val_mse,
                    'val_rmse': val_rmse
                }
                self.tuning_results.append(result)

                # Update best model based on validation RMSE
                if val_rmse < self.best_aic:
                    self.best_aic = val_rmse
                    self.best_params = (p, d, q)
                    self.best_model = fitted_model
                    self.logger.info(f"  [{idx}/{len(param_combinations)}] New best model found!")
                    self.logger.info(f"    Order: {(p, d, q)}, Val RMSE: {val_rmse:.2f}, AIC: {fitted_model.aic:.2f}")

                if idx % 5 == 0:
                    self.logger.info(f"  Progress: {idx}/{len(param_combinations)} combinations tested")

            except Exception as e:
                self.logger.warning(f"  Failed for order {(p, d, q)}: {str(e)}")
                continue

        self.logger.info("\n" + "-"*80)
        self.logger.info("HYPERPARAMETER TUNING COMPLETED")
        self.logger.info(f"Best parameters: {self.best_params}")
        self.logger.info(f"Best validation RMSE: {self.best_aic:.2f}")
        self.logger.info("-"*80)

        return {
            'best_order': self.best_params,
            'best_val_rmse': self.best_aic,
            'all_results': self.tuning_results
        }


# ============================================================================
# MODEL TRAINING
# ============================================================================

def train_arimax_model(train_endog: pd.Series,
                      train_exog: np.ndarray,
                      order: Tuple[int, int, int],
                      logger: logging.Logger) -> SARIMAX:
    """
    Train ARIMAX model with best parameters.

    Args:
        train_endog: Training target variable
        train_exog: Training exogenous features
        order: ARIMAX order (p, d, q)
        logger: Logger instance

    Returns:
        Fitted SARIMAX model
    """
    logger.info("\n" + "="*80)
    logger.info("MODEL TRAINING WITH BEST PARAMETERS")
    logger.info("="*80)

    logger.info(f"\nTraining ARIMAX{order} model...")
    logger.info(f"Training samples: {len(train_endog)}")
    logger.info(f"Exogenous features: {train_exog.shape[1]}")

    model = SARIMAX(train_endog,
                   exog=train_exog,
                   order=order,
                   enforce_stationarity=False,
                   enforce_invertibility=False)

    fitted_model = model.fit(disp=False, maxiter=200)

    logger.info("\nModel training completed successfully!")
    logger.info(f"AIC: {fitted_model.aic:.2f}")
    logger.info(f"BIC: {fitted_model.bic:.2f}")
    logger.info(f"Log-Likelihood: {fitted_model.llf:.2f}")

    return fitted_model


# ============================================================================
# MODEL EVALUATION
# ============================================================================

class ARIMAXEvaluator:
    """
    Comprehensive evaluation of ARIMAX models.
    """

    def __init__(self, logger: logging.Logger):
        self.logger = logger
        self.metrics = {}

    def calculate_metrics(self, y_true: np.ndarray, y_pred: np.ndarray,
                         dataset_name: str = "Dataset") -> Dict:
        """
        Calculate comprehensive evaluation metrics.

        Args:
            y_true: True values
            y_pred: Predicted values
            dataset_name: Name of the dataset (Train/Val/Test)

        Returns:
            Dictionary of metrics
        """
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mape = mean_absolute_percentage_error(y_true, y_pred) * 100
        r2 = r2_score(y_true, y_pred)

        metrics = {
            'MSE': mse,
            'RMSE': rmse,
            'MAPE': mape,
            'R2': r2
        }

        self.logger.info(f"\n{dataset_name} Metrics:")
        self.logger.info(f"  MSE:  {mse:.2f}")
        self.logger.info(f"  RMSE: {rmse:.2f}")
        self.logger.info(f"  MAPE: {mape:.2f}%")
        self.logger.info(f"  R²:   {r2:.4f}")

        return metrics

    def time_series_cross_validation(self,
                                    endog: pd.Series,
                                    exog: np.ndarray,
                                    order: Tuple[int, int, int],
                                    n_splits: int = 5) -> Dict:
        """
        Perform time series cross-validation.

        Args:
            endog: Endogenous variable (target)
            exog: Exogenous variables (features)
            order: ARIMAX order
            n_splits: Number of cross-validation splits

        Returns:
            Dictionary with CV metrics
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("TIME SERIES CROSS-VALIDATION")
        self.logger.info("="*80)

        self.logger.info(f"\nPerforming {n_splits}-fold time series cross-validation...")

        n = len(endog)
        fold_size = n // (n_splits + 1)

        cv_scores = {
            'rmse': [],
            'mse': [],
            'mape': [],
            'r2': []
        }

        for i in range(n_splits):
            train_end = fold_size * (i + 2)
            test_start = train_end
            test_end = min(train_end + fold_size, n)

            if test_end <= test_start:
                break

            # Split data
            train_endog = endog[:train_end]
            train_exog = exog[:train_end]
            test_endog = endog[test_start:test_end]
            test_exog = exog[test_start:test_end]

            try:
                # Train model
                model = SARIMAX(train_endog, exog=train_exog, order=order,
                              enforce_stationarity=False, enforce_invertibility=False)
                fitted = model.fit(disp=False, maxiter=200)

                # Predict
                predictions = fitted.forecast(steps=len(test_endog), exog=test_exog)

                # Calculate metrics
                mse = mean_squared_error(test_endog, predictions)
                rmse = np.sqrt(mse)
                mape = mean_absolute_percentage_error(test_endog, predictions) * 100
                r2 = r2_score(test_endog, predictions)

                cv_scores['mse'].append(mse)
                cv_scores['rmse'].append(rmse)
                cv_scores['mape'].append(mape)
                cv_scores['r2'].append(r2)

                self.logger.info(f"\nFold {i+1}/{n_splits}:")
                self.logger.info(f"  Train size: {len(train_endog)}, Test size: {len(test_endog)}")
                self.logger.info(f"  RMSE: {rmse:.2f}, MAPE: {mape:.2f}%, R²: {r2:.4f}")

            except Exception as e:
                self.logger.warning(f"Fold {i+1} failed: {str(e)}")
                continue

        # Calculate average scores
        cv_results = {
            'mean_rmse': np.mean(cv_scores['rmse']),
            'std_rmse': np.std(cv_scores['rmse']),
            'mean_mse': np.mean(cv_scores['mse']),
            'std_mse': np.std(cv_scores['mse']),
            'mean_mape': np.mean(cv_scores['mape']),
            'std_mape': np.std(cv_scores['mape']),
            'mean_r2': np.mean(cv_scores['r2']),
            'std_r2': np.std(cv_scores['r2'])
        }

        self.logger.info("\n" + "-"*80)
        self.logger.info("CROSS-VALIDATION RESULTS:")
        self.logger.info(f"  Mean RMSE: {cv_results['mean_rmse']:.2f} ± {cv_results['std_rmse']:.2f}")
        self.logger.info(f"  Mean MAPE: {cv_results['mean_mape']:.2f}% ± {cv_results['std_mape']:.2f}%")
        self.logger.info(f"  Mean R²:   {cv_results['mean_r2']:.4f} ± {cv_results['std_r2']:.4f}")
        self.logger.info("-"*80)

        return cv_results

    def evaluate_model(self,
                      model: SARIMAX,
                      train_endog: pd.Series, train_exog: np.ndarray,
                      val_endog: pd.Series, val_exog: np.ndarray,
                      test_endog: pd.Series, test_exog: np.ndarray) -> Dict:
        """
        Comprehensive model evaluation on all datasets.

        Args:
            model: Fitted SARIMAX model
            train_endog, train_exog: Training data
            val_endog, val_exog: Validation data
            test_endog, test_exog: Test data

        Returns:
            Dictionary with all evaluation metrics
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("COMPREHENSIVE MODEL EVALUATION")
        self.logger.info("="*80)

        # Training set evaluation
        train_pred = model.predict(start=0, end=len(train_endog)-1, exog=train_exog)
        train_metrics = self.calculate_metrics(train_endog, train_pred, "Training Set")

        # Validation set evaluation
        val_pred = model.forecast(steps=len(val_endog), exog=val_exog)
        val_metrics = self.calculate_metrics(val_endog, val_pred, "Validation Set")

        # Test set evaluation
        test_pred = model.forecast(steps=len(test_endog), exog=test_exog)
        test_metrics = self.calculate_metrics(test_endog, test_pred, "Test Set")

        all_metrics = {
            'train': train_metrics,
            'validation': val_metrics,
            'test': test_metrics
        }

        self.metrics = all_metrics

        return all_metrics


# ============================================================================
# RESULTS SAVING
# ============================================================================

def save_results(model: SARIMAX,
                metrics: Dict,
                cv_results: Dict,
                tuning_results: Dict,
                scaler,
                output_dir: str = 'results',
                logger: logging.Logger = None):
    """
    Save model, metrics, and results to disk.

    Args:
        model: Fitted SARIMAX model
        metrics: Evaluation metrics
        cv_results: Cross-validation results
        tuning_results: Hyperparameter tuning results
        scaler: Fitted scaler for exogenous features
        output_dir: Directory to save results
        logger: Logger instance
    """
    logger.info("\n" + "="*80)
    logger.info("SAVING RESULTS")
    logger.info("="*80)

    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Save model
    model_path = os.path.join(output_dir, f'arimax_model_{timestamp}.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    logger.info(f"\nModel saved: {model_path}")

    # Save scaler
    scaler_path = os.path.join(output_dir, f'exog_scaler_{timestamp}.pkl')
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    logger.info(f"Scaler saved: {scaler_path}")

    # Save metrics
    metrics_path = os.path.join(output_dir, f'metrics_{timestamp}.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)
    logger.info(f"Metrics saved: {metrics_path}")

    # Save CV results
    cv_path = os.path.join(output_dir, f'cv_results_{timestamp}.json')
    with open(cv_path, 'w') as f:
        json.dump(cv_results, f, indent=4)
    logger.info(f"CV results saved: {cv_path}")

    # Save tuning results
    tuning_path = os.path.join(output_dir, f'tuning_results_{timestamp}.json')
    # Convert to JSON-serializable format
    tuning_serializable = {
        'best_order': list(tuning_results['best_order']),
        'best_val_rmse': float(tuning_results['best_val_rmse']),
        'all_results': [
            {
                'order': list(r['order']),
                'aic': float(r['aic']),
                'bic': float(r['bic']),
                'val_mse': float(r['val_mse']),
                'val_rmse': float(r['val_rmse'])
            }
            for r in tuning_results['all_results']
        ]
    }
    with open(tuning_path, 'w') as f:
        json.dump(tuning_serializable, f, indent=4)
    logger.info(f"Tuning results saved: {tuning_path}")

    # Save summary report
    summary_path = os.path.join(output_dir, f'model_summary_{timestamp}.txt')
    with open(summary_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("ARIMAX MODEL EXPLORATION - SUMMARY REPORT\n")
        f.write("="*80 + "\n\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(f"Best Model Order: {tuning_results['best_order']}\n\n")
        f.write("EVALUATION METRICS:\n")
        f.write("-"*80 + "\n")
        for dataset, dataset_metrics in metrics.items():
            f.write(f"\n{dataset.upper()}:\n")
            for metric, value in dataset_metrics.items():
                f.write(f"  {metric}: {value:.4f}\n")
        f.write("\n" + "-"*80 + "\n")
        f.write("\nCROSS-VALIDATION RESULTS:\n")
        f.write("-"*80 + "\n")
        for metric, value in cv_results.items():
            f.write(f"  {metric}: {value:.4f}\n")
    logger.info(f"Summary report saved: {summary_path}")

    logger.info("\n" + "="*80)
    logger.info("ALL RESULTS SAVED SUCCESSFULLY")
    logger.info("="*80)


# ============================================================================
# MAIN EXECUTION PIPELINE
# ============================================================================

def main():
    """
    Main execution pipeline for ARIMAX model exploration.
    """
    # Setup logging
    logger = setup_logging()

    try:
        # 1. Initialize data processor
        logger.info("\nInitializing ARIMAX data processor...")
        processor = ARIMAXDataProcessor(logger)

        # 2. Load data
        df = processor.load_data('preprocessed-dataset.csv')

        # 3. Check stationarity of target variable
        processor.check_stationarity(df['arrivals'], "Tourist Arrivals")

        # 4. Feature preparation
        df, exog_features = processor.prepare_features(df)

        # 5. Time-series split
        train_df, val_df, test_df, _, _, _ = time_series_split(
            df, train_ratio=0.75, val_ratio=0.15, logger=logger
        )

        # 6. Prepare arrays
        train_endog = train_df['arrivals']
        train_exog = train_df[exog_features].values

        val_endog = val_df['arrivals']
        val_exog = val_df[exog_features].values

        test_endog = test_df['arrivals']
        test_exog = test_df[exog_features].values

        # 7. Scale exogenous features
        train_exog_scaled, val_exog_scaled, test_exog_scaled = processor.scale_exogenous_features(
            train_exog, val_exog, test_exog
        )

        # 8. Hyperparameter tuning
        tuner = ARIMAXTuner(logger)
        param_combinations = tuner.generate_param_combinations(
            p_range=range(0, 4),
            d_range=range(0, 2),
            q_range=range(0, 4)
        )

        tuning_results = tuner.tune_hyperparameters(
            train_endog, train_exog_scaled,
            val_endog, val_exog_scaled,
            param_combinations
        )

        # 9. Train final model with best parameters
        best_order = tuning_results['best_order']
        final_model = train_arimax_model(
            train_endog, train_exog_scaled, best_order, logger
        )

        # 10. Evaluate model
        evaluator = ARIMAXEvaluator(logger)
        all_metrics = evaluator.evaluate_model(
            final_model,
            train_endog, train_exog_scaled,
            val_endog, val_exog_scaled,
            test_endog, test_exog_scaled
        )

        # 11. Time series cross-validation
        full_endog = pd.concat([train_endog, val_endog])
        full_exog = np.vstack([train_exog_scaled, val_exog_scaled])

        cv_results = evaluator.time_series_cross_validation(
            full_endog, full_exog, best_order, n_splits=5
        )

        # 12. Save all results
        save_results(
            final_model, all_metrics, cv_results, tuning_results,
            processor.exog_scaler, output_dir='results', logger=logger
        )

        logger.info("\n" + "="*80)
        logger.info("ARIMAX MODEL EXPLORATION COMPLETED SUCCESSFULLY")
        logger.info("="*80)

    except Exception as e:
        logger.error(f"\n\nERROR: {str(e)}", exc_info=True)
        raise


if __name__ == "__main__":
    main()

2026-03-17 08:16:55 - ARIMAX_Explorer - INFO - ================================================================================
INFO:ARIMAX_Explorer:================================================================================
2026-03-17 08:16:55 - ARIMAX_Explorer - INFO - ARIMAX MODEL EXPLORATION INITIATED
INFO:ARIMAX_Explorer:ARIMAX MODEL EXPLORATION INITIATED
2026-03-17 08:16:55 - ARIMAX_Explorer - INFO - ================================================================================
INFO:ARIMAX_Explorer:================================================================================
2026-03-17 08:16:55 - ARIMAX_Explorer - INFO - 
Initializing ARIMAX data processor...
INFO:ARIMAX_Explorer:
Initializing ARIMAX data processor...
2026-03-17 08:16:55 - ARIMAX_Explorer - INFO - Loading data from: preprocessed-dataset.csv
INFO:ARIMAX_Explorer:Loading data from: preprocessed-dataset.csv
2026-03-17 08:16:55 - ARIMAX_Explorer - INFO - Data loaded successfully: 5740 rows, 20 columns
INFO:

In [ ]:
"""
SARIMAX Model Exploration for Sri Lankan Tourism Prediction
============================================================
Author: ML Research Team
Date: December 2025 (Optimized: March 2026)
Description: Production-ready SARIMAX implementation with seasonal components,
             proper time-series validation, hyperparameter tuning, and comprehensive evaluation.

OPTIMIZATIONS APPLIED:
- Reduced hyperparameter search space (default ranges now generate ~72 combinations instead of 144)
- Default max_combinations reduced from 50 → 25 (random sample when needed)
- maxiter reduced from 200 → 100 across ALL model fits (training, tuning, CV)
- Time-series CV splits reduced from 5 → 3
- These changes typically cut runtime by 50-70% while preserving model quality and exploration.
"""

import os
import logging
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Tuple, Dict, List

# Statistical and ML Libraries
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import itertools
import json
import pickle
import random  # Already used in tuner - kept for clarity

warnings.filterwarnings('ignore')

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

def setup_logging(log_dir: str = 'logs') -> logging.Logger:
    """
    Setup comprehensive logging for model exploration.
    """
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = os.path.join(log_dir, f'sarimax_exploration_{timestamp}.log')

    logger = logging.getLogger('SARIMAX_Explorer')
    logger.setLevel(logging.INFO)

    fh = logging.FileHandler(log_file)
    fh.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)

    formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)

    logger.addHandler(fh)
    logger.addHandler(ch)

    logger.info("="*80)
    logger.info("SARIMAX MODEL EXPLORATION INITIATED (OPTIMIZED VERSION)")
    logger.info("="*80)

    return logger


# ============================================================================
# DATA LOADING AND PREPROCESSING
# ============================================================================

class SARIMAXDataProcessor:
    """
    SARIMAX-specific data processing including feature engineering,
    seasonal analysis, and scaling.
    """

    def __init__(self, logger: logging.Logger):
        self.logger = logger
        self.exog_scaler = StandardScaler()
        self.feature_columns = None
        self.target_column = 'arrivals'
        self.seasonal_period = None

    def load_data(self, file_path: str) -> pd.DataFrame:
        """Load and validate the preprocessed dataset."""
        self.logger.info(f"Loading data from: {file_path}")

        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)

        self.logger.info(f"Data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")
        self.logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")

        return df

    def check_stationarity(self, series: pd.Series, name: str = "Series") -> bool:
        result = adfuller(series.dropna())
        self.logger.info(f"\nStationarity Test - {name}:")
        self.logger.info(f"  ADF Statistic: {result[0]:.6f}")
        self.logger.info(f"  p-value: {result[1]:.6f}")
        self.logger.info(f"  Critical Values:")
        for key, value in result[4].items():
            self.logger.info(f"    {key}: {value:.3f}")

        is_stationary = result[1] < 0.05
        self.logger.info(f"  Stationary: {is_stationary}")
        return is_stationary

    def analyze_seasonality(self, df: pd.DataFrame, period: int = 7) -> Dict:
        self.logger.info("\n" + "="*80)
        self.logger.info("SEASONAL DECOMPOSITION ANALYSIS")
        self.logger.info("="*80)

        ts_data = df.set_index('date')['arrivals']

        self.logger.info(f"\nAnalyzing seasonality with period = {period}")

        try:
            decomposition = seasonal_decompose(ts_data, model='additive', period=period, extrapolate_trend='freq')

            seasonal_strength = 1 - (np.var(decomposition.resid.dropna()) /
                                    np.var(decomposition.seasonal.dropna() + decomposition.resid.dropna()))

            trend_strength = 1 - (np.var(decomposition.resid.dropna()) /
                                 np.var(decomposition.trend.dropna() + decomposition.resid.dropna()))

            self.logger.info(f"\nSeasonality Analysis Results:")
            self.logger.info(f"  Seasonal Strength: {seasonal_strength:.4f}")
            self.logger.info(f"  Trend Strength: {trend_strength:.4f}")
            self.logger.info(f"  Residual Variance: {np.var(decomposition.resid.dropna()):.2f}")

            if seasonal_strength > 0.6:
                self.logger.info(f"  → Strong seasonality detected (period={period})")
            elif seasonal_strength > 0.3:
                self.logger.info(f"  → Moderate seasonality detected (period={period})")
            else:
                self.logger.info(f"  → Weak seasonality (period={period})")

            self.seasonal_period = period

            return {
                'period': period,
                'seasonal_strength': seasonal_strength,
                'trend_strength': trend_strength,
                'residual_variance': float(np.var(decomposition.resid.dropna()))
            }

        except Exception as e:
            self.logger.warning(f"Could not perform seasonal decomposition: {str(e)}")
            return {'period': period, 'seasonal_strength': 0.0}

    def prepare_features(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
        self.logger.info("\n" + "="*80)
        self.logger.info("FEATURE PREPARATION FOR SARIMAX")
        self.logger.info("="*80)

        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        exog_features = [col for col in df.columns if col not in exclude_cols]

        self.logger.info(f"\nBase exogenous features: {len(exog_features)}")

        # Create lagged features (kept as-is)
        lag_features = ['usd_lkr', 'web_search', 'image_search', 'temperature', 'event_encoded']
        self.logger.info(f"\nCreating lagged features for: {lag_features}")

        for feature in lag_features:
            if feature in df.columns:
                df[f'{feature}_lag1'] = df[feature].shift(1)
                df[f'{feature}_lag7'] = df[feature].shift(7)
                df[f'{feature}_lag14'] = df[feature].shift(14)
                exog_features.extend([f'{feature}_lag1', f'{feature}_lag7', f'{feature}_lag14'])

        # Create rolling statistics (kept as-is)
        self.logger.info("\nCreating rolling statistics features...")
        window_sizes = [7, 14, 30]
        rolling_features = ['temperature', 'humidity', 'web_search', 'image_search']

        for feature in rolling_features:
            if feature in df.columns:
                for window in window_sizes:
                    df[f'{feature}_rolling_mean_{window}'] = df[feature].rolling(window=window, min_periods=1).mean()
                    df[f'{feature}_rolling_std_{window}'] = df[feature].rolling(window=window, min_periods=1).std()
                    exog_features.extend([
                        f'{feature}_rolling_mean_{window}',
                        f'{feature}_rolling_std_{window}'
                    ])

        # Add temporal features (kept as-is)
        self.logger.info("\nCreating temporal and seasonal features...")
        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['day_of_year'] = df['date'].dt.dayofyear
        df['week_of_year'] = df['date'].dt.isocalendar().week
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
        df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
        df['is_quarter_start'] = df['date'].dt.is_quarter_start.astype(int)

        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
        df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)

        df['is_high_season'] = df['month'].isin([12, 1, 2, 7, 8]).astype(int)
        df['is_low_season'] = df['month'].isin([5, 6, 9, 10]).astype(int)

        temporal_features = [
            'day_of_week', 'day_of_month', 'month', 'quarter', 'day_of_year',
            'week_of_year', 'is_weekend', 'is_month_start', 'is_month_end',
            'is_quarter_start', 'month_sin', 'month_cos', 'day_sin', 'day_cos',
            'week_sin', 'week_cos', 'is_high_season', 'is_low_season'
        ]
        exog_features.extend(temporal_features)

        # Handle missing values
        self.logger.info("\nHandling missing values...")
        initial_missing = df[exog_features].isnull().sum().sum()
        df[exog_features] = df[exog_features].fillna(method='ffill').fillna(method='bfill')
        final_missing = df[exog_features].isnull().sum().sum()
        self.logger.info(f"Missing values before: {initial_missing}, after: {final_missing}")

        self.feature_columns = exog_features
        self.logger.info(f"\nTotal exogenous features: {len(exog_features)}")

        return df, exog_features

    def scale_exogenous_features(self, train_exog: np.ndarray,
                                val_exog: np.ndarray,
                                test_exog: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        self.logger.info("\nScaling exogenous features for numerical stability...")

        train_scaled = self.exog_scaler.fit_transform(train_exog)
        val_scaled = self.exog_scaler.transform(val_exog)
        test_scaled = self.exog_scaler.transform(test_exog)

        self.logger.info(f"Scaling completed - Mean: {train_scaled.mean():.4f}, Std: {train_scaled.std():.4f}")

        return train_scaled, val_scaled, test_scaled


# ============================================================================
# DATA SPLITTING
# ============================================================================

def time_series_split(df: pd.DataFrame,
                     train_ratio: float = 0.75,
                     val_ratio: float = 0.15,
                     logger: logging.Logger = None) -> Tuple:
    logger.info("\n" + "="*80)
    logger.info("TIME-SERIES AWARE DATA SPLITTING")
    logger.info("="*80)

    n = len(df)
    train_size = int(n * train_ratio)
    val_size = int(n * val_ratio)

    train_df = df.iloc[:train_size].copy()
    val_df = df.iloc[train_size:train_size + val_size].copy()
    test_df = df.iloc[train_size + val_size:].copy()

    logger.info(f"\nTotal samples: {n}")
    logger.info(f"Train: {len(train_df)} samples ({train_ratio*100:.1f}%)")
    logger.info(f"Validation: {len(val_df)} samples ({val_ratio*100:.1f}%)")
    logger.info(f"Test: {len(test_df)} samples ({(1-train_ratio-val_ratio)*100:.1f}%)")

    return (train_df, val_df, test_df,
            range(len(train_df)),
            range(len(train_df), len(train_df) + len(val_df)),
            range(len(train_df) + len(val_df), n))


# ============================================================================
# HYPERPARAMETER TUNING (OPTIMIZED)
# ============================================================================

class SARIMAXTuner:
    """
    Hyperparameter tuning for SARIMAX models using grid search (optimized ranges).
    """

    def __init__(self, logger: logging.Logger, seasonal_period: int = 7):
        self.logger = logger
        self.seasonal_period = seasonal_period
        self.best_params = None
        self.best_seasonal_params = None
        self.best_model = None
        self.best_score = np.inf
        self.tuning_results = []

    def generate_param_combinations(self,
                                    p_range: range = range(0, 3),
                                    d_range: range = range(0, 2),
                                    q_range: range = range(0, 3),
                                    P_range: range = range(0, 2),
                                    D_range: range = range(0, 2),
                                    Q_range: range = range(0, 1)) -> List[Tuple]:
        """
        Generate parameter combinations for SARIMAX grid search.

        OPTIMIZATION: Q_range limited to 0-1 (common for tourism data).
        Total combinations now ~72 instead of 144.
        """
        non_seasonal = list(itertools.product(p_range, d_range, q_range))
        seasonal = list(itertools.product(P_range, D_range, Q_range))

        param_combinations = [(ns, s) for ns in non_seasonal for s in seasonal]

        self.logger.info(f"\nGenerated {len(param_combinations)} parameter combinations")
        self.logger.info(f"Non-seasonal params: {len(non_seasonal)}")
        self.logger.info(f"Seasonal params: {len(seasonal)}")
        self.logger.info(f"Seasonal period: {self.seasonal_period}")

        return param_combinations

    def tune_hyperparameters(self,
                           train_endog: pd.Series,
                           train_exog: np.ndarray,
                           val_endog: pd.Series,
                           val_exog: np.ndarray,
                           param_combinations: List[Tuple],
                           max_combinations: int = 25) -> Dict:
        """
        Perform grid search for hyperparameter tuning (max 25 combinations).
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("HYPERPARAMETER TUNING - GRID SEARCH FOR SARIMAX")
        self.logger.info("="*80)

        if len(param_combinations) > max_combinations:
            self.logger.info(f"\nLimiting search to {max_combinations} combinations (random sample)")
            random.seed(42)
            param_combinations = random.sample(param_combinations, max_combinations)

        self.logger.info(f"\nTesting {len(param_combinations)} parameter combinations...")

        for idx, (order, seasonal_order) in enumerate(param_combinations, 1):
            try:
                full_seasonal_order = seasonal_order + (self.seasonal_period,)

                model = SARIMAX(train_endog,
                              exog=train_exog,
                              order=order,
                              seasonal_order=full_seasonal_order,
                              enforce_stationarity=False,
                              enforce_invertibility=False)

                fitted_model = model.fit(disp=False, maxiter=100, method='lbfgs')  # OPTIMIZED: maxiter=100

                val_predictions = fitted_model.forecast(steps=len(val_endog), exog=val_exog)
                val_mse = mean_squared_error(val_endog, val_predictions)
                val_rmse = np.sqrt(val_mse)

                result = {
                    'order': order,
                    'seasonal_order': full_seasonal_order,
                    'aic': fitted_model.aic,
                    'bic': fitted_model.bic,
                    'val_mse': val_mse,
                    'val_rmse': val_rmse
                }
                self.tuning_results.append(result)

                if val_rmse < self.best_score:
                    self.best_score = val_rmse
                    self.best_params = order
                    self.best_seasonal_params = full_seasonal_order
                    self.best_model = fitted_model
                    self.logger.info(f"  [{idx}/{len(param_combinations)}] New best model found!")
                    self.logger.info(f"    Order: {order}x{full_seasonal_order}")
                    self.logger.info(f"    Val RMSE: {val_rmse:.2f}, AIC: {fitted_model.aic:.2f}")

                if idx % 5 == 0:
                    self.logger.info(f"  Progress: {idx}/{len(param_combinations)} combinations tested")

            except Exception as e:
                self.logger.warning(f"  Failed for order {order}x{seasonal_order}: {str(e)[:100]}")
                continue

        self.logger.info("\n" + "-"*80)
        self.logger.info("HYPERPARAMETER TUNING COMPLETED")
        self.logger.info(f"Best non-seasonal order: {self.best_params}")
        self.logger.info(f"Best seasonal order: {self.best_seasonal_params}")
        self.logger.info(f"Best validation RMSE: {self.best_score:.2f}")
        self.logger.info("-"*80)

        return {
            'best_order': self.best_params,
            'best_seasonal_order': self.best_seasonal_params,
            'best_val_rmse': self.best_score,
            'all_results': self.tuning_results
        }


# ============================================================================
# MODEL TRAINING (OPTIMIZED)
# ============================================================================

def train_sarimax_model(train_endog: pd.Series,
                       train_exog: np.ndarray,
                       order: Tuple[int, int, int],
                       seasonal_order: Tuple[int, int, int, int],
                       logger: logging.Logger) -> SARIMAX:
    logger.info("\n" + "="*80)
    logger.info("MODEL TRAINING WITH BEST PARAMETERS")
    logger.info("="*80)

    logger.info(f"\nTraining SARIMAX{order}x{seasonal_order} model...")
    logger.info(f"Training samples: {len(train_endog)}")
    logger.info(f"Exogenous features: {train_exog.shape[1]}")

    model = SARIMAX(train_endog,
                   exog=train_exog,
                   order=order,
                   seasonal_order=seasonal_order,
                   enforce_stationarity=False,
                   enforce_invertibility=False)

    fitted_model = model.fit(disp=False, maxiter=100, method='lbfgs')  # OPTIMIZED: maxiter=100

    logger.info("\nModel training completed successfully!")
    logger.info(f"AIC: {fitted_model.aic:.2f}")
    logger.info(f"BIC: {fitted_model.bic:.2f}")

    return fitted_model


# ============================================================================
# MODEL EVALUATION
# ============================================================================

class SARIMAXEvaluator:
    """
    Comprehensive evaluation of SARIMAX models.
    """

    def __init__(self, logger: logging.Logger):
        self.logger = logger
        self.metrics = {}

    def calculate_metrics(self, y_true: np.ndarray, y_pred: np.ndarray,
                         dataset_name: str = "Dataset") -> Dict:
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mape = mean_absolute_percentage_error(y_true, y_pred) * 100
        r2 = r2_score(y_true, y_pred)
        mae = np.mean(np.abs(y_true - y_pred))

        metrics = {
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'MAPE': mape,
            'R2': r2
        }

        self.logger.info(f"\n{dataset_name} Metrics:")
        self.logger.info(f"  MSE:  {mse:.2f}")
        self.logger.info(f"  RMSE: {rmse:.2f}")
        self.logger.info(f"  MAE:  {mae:.2f}")
        self.logger.info(f"  MAPE: {mape:.2f}%")
        self.logger.info(f"  R²:   {r2:.4f}")

        return metrics

    def time_series_cross_validation(self,
                                    endog: pd.Series,
                                    exog: np.ndarray,
                                    order: Tuple[int, int, int],
                                    seasonal_order: Tuple[int, int, int, int],
                                    n_splits: int = 3) -> Dict:  # OPTIMIZED: 3 splits instead of 5
        self.logger.info("\n" + "="*80)
        self.logger.info("TIME SERIES CROSS-VALIDATION")
        self.logger.info("="*80)

        self.logger.info(f"\nPerforming {n_splits}-fold time series cross-validation...")

        n = len(endog)
        fold_size = n // (n_splits + 1)

        cv_scores = {'rmse': [], 'mse': [], 'mae': [], 'mape': [], 'r2': []}

        for i in range(n_splits):
            train_end = fold_size * (i + 2)
            test_start = train_end
            test_end = min(train_end + fold_size, n)

            if test_end <= test_start:
                break

            train_endog = endog[:train_end]
            train_exog = exog[:train_end]
            test_endog = endog[test_start:test_end]
            test_exog = exog[test_start:test_end]

            try:
                model = SARIMAX(train_endog, exog=train_exog,
                              order=order, seasonal_order=seasonal_order,
                              enforce_stationarity=False, enforce_invertibility=False)
                fitted = model.fit(disp=False, maxiter=100, method='lbfgs')  # OPTIMIZED: maxiter=100

                predictions = fitted.forecast(steps=len(test_endog), exog=test_exog)

                mse = mean_squared_error(test_endog, predictions)
                rmse = np.sqrt(mse)
                mae = np.mean(np.abs(test_endog - predictions))
                mape = mean_absolute_percentage_error(test_endog, predictions) * 100
                r2 = r2_score(test_endog, predictions)

                cv_scores['mse'].append(mse)
                cv_scores['rmse'].append(rmse)
                cv_scores['mae'].append(mae)
                cv_scores['mape'].append(mape)
                cv_scores['r2'].append(r2)

                self.logger.info(f"\nFold {i+1}/{n_splits}: RMSE: {rmse:.2f}, MAPE: {mape:.2f}%")

            except Exception as e:
                self.logger.warning(f"Fold {i+1} failed: {str(e)}")
                continue

        cv_results = {
            'mean_rmse': np.mean(cv_scores['rmse']),
            'std_rmse': np.std(cv_scores['rmse']),
            'mean_mse': np.mean(cv_scores['mse']),
            'std_mse': np.std(cv_scores['mse']),
            'mean_mae': np.mean(cv_scores['mae']),
            'std_mae': np.std(cv_scores['mae']),
            'mean_mape': np.mean(cv_scores['mape']),
            'std_mape': np.std(cv_scores['mape']),
            'mean_r2': np.mean(cv_scores['r2']),
            'std_r2': np.std(cv_scores['r2']),
            'n_folds_completed': len(cv_scores['rmse'])
        }

        self.logger.info("\n" + "-"*80)
        self.logger.info("CROSS-VALIDATION RESULTS:")
        self.logger.info(f"  Mean RMSE: {cv_results['mean_rmse']:.2f} ± {cv_results['std_rmse']:.2f}")
        self.logger.info(f"  Mean MAPE: {cv_results['mean_mape']:.2f}%")
        self.logger.info("-"*80)

        return cv_results

    def evaluate_model(self,
                      model: SARIMAX,
                      train_endog: pd.Series, train_exog: np.ndarray,
                      val_endog: pd.Series, val_exog: np.ndarray,
                      test_endog: pd.Series, test_exog: np.ndarray) -> Dict:
        self.logger.info("\n" + "="*80)
        self.logger.info("COMPREHENSIVE MODEL EVALUATION")
        self.logger.info("="*80)

        train_pred = model.predict(start=0, end=len(train_endog)-1, exog=train_exog)
        train_metrics = self.calculate_metrics(train_endog, train_pred, "Training Set")

        val_pred = model.forecast(steps=len(val_endog), exog=val_exog)
        val_metrics = self.calculate_metrics(val_endog, val_pred, "Validation Set")

        test_pred = model.forecast(steps=len(test_endog), exog=test_exog)
        test_metrics = self.calculate_metrics(test_endog, test_pred, "Test Set")

        all_metrics = {
            'train': train_metrics,
            'validation': val_metrics,
            'test': test_metrics
        }

        self.metrics = all_metrics
        return all_metrics


# ============================================================================
# RESULTS SAVING
# ============================================================================

def save_results(model: SARIMAX,
                metrics: Dict,
                cv_results: Dict,
                tuning_results: Dict,
                seasonality_analysis: Dict,
                scaler,
                output_dir: str = 'results',
                logger: logging.Logger = None):
    logger.info("\n" + "="*80)
    logger.info("SAVING RESULTS")
    logger.info("="*80)

    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    model_path = os.path.join(output_dir, f'sarimax_model_{timestamp}.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    logger.info(f"Model saved: {model_path}")

    scaler_path = os.path.join(output_dir, f'exog_scaler_{timestamp}.pkl')
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    logger.info(f"Scaler saved: {scaler_path}")

    # Save other artifacts (metrics, CV, seasonality, tuning) - unchanged but kept for completeness
    metrics_path = os.path.join(output_dir, f'metrics_{timestamp}.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)

    cv_path = os.path.join(output_dir, f'cv_results_{timestamp}.json')
    with open(cv_path, 'w') as f:
        json.dump(cv_results, f, indent=4)

    seasonality_path = os.path.join(output_dir, f'seasonality_analysis_{timestamp}.json')
    with open(seasonality_path, 'w') as f:
        json.dump(seasonality_analysis, f, indent=4)

    tuning_path = os.path.join(output_dir, f'tuning_results_{timestamp}.json')
    tuning_serializable = {
        'best_order': list(tuning_results['best_order']),
        'best_seasonal_order': list(tuning_results['best_seasonal_order']),
        'best_val_rmse': float(tuning_results['best_val_rmse']),
        'all_results': [
            {
                'order': list(r['order']),
                'seasonal_order': list(r['seasonal_order']),
                'aic': float(r['aic']),
                'bic': float(r['bic']),
                'val_mse': float(r['val_mse']),
                'val_rmse': float(r['val_rmse'])
            }
            for r in tuning_results['all_results']
        ]
    }
    with open(tuning_path, 'w') as f:
        json.dump(tuning_serializable, f, indent=4)

    # Summary report
    summary_path = os.path.join(output_dir, f'model_summary_{timestamp}.txt')
    with open(summary_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("SARIMAX MODEL EXPLORATION - SUMMARY REPORT (OPTIMIZED)\n")
        f.write("="*80 + "\n\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(f"Best Model Order: {tuning_results['best_order']}\n")
        f.write(f"Best Seasonal Order: {tuning_results['best_seasonal_order']}\n\n")
        f.write("EVALUATION METRICS:\n")
        for dataset, dataset_metrics in metrics.items():
            f.write(f"\n{dataset.upper()}:\n")
            for metric, value in dataset_metrics.items():
                f.write(f"  {metric}: {value:.4f}\n")
    logger.info(f"Summary report saved: {summary_path}")

    logger.info("\nALL RESULTS SAVED SUCCESSFULLY")


# ============================================================================
# MAIN EXECUTION PIPELINE
# ============================================================================

def main():
    logger = setup_logging()

    try:
        processor = SARIMAXDataProcessor(logger)
        df = processor.load_data('preprocessed-dataset.csv')

        # Seasonality analysis
        seasonality_results = {}
        for period in [7, 30]:
            season_stats = processor.analyze_seasonality(df, period=period)
            seasonality_results[f'period_{period}'] = season_stats

        best_period = 7
        best_strength = 0
        for period_key, stats in seasonality_results.items():
            if stats.get('seasonal_strength', 0) > best_strength:
                best_strength = stats['seasonal_strength']
                best_period = stats['period']

        processor.seasonal_period = best_period
        logger.info(f"\nSelected seasonal period: {best_period} (strength: {best_strength:.4f})")

        processor.check_stationarity(df['arrivals'], "Tourist Arrivals")
        df, exog_features = processor.prepare_features(df)

        train_df, val_df, test_df, _, _, _ = time_series_split(df, train_ratio=0.75, val_ratio=0.15, logger=logger)

        train_endog = train_df['arrivals']
        train_exog = train_df[exog_features].values
        val_endog = val_df['arrivals']
        val_exog = val_df[exog_features].values
        test_endog = test_df['arrivals']
        test_exog = test_df[exog_features].values

        train_exog_scaled, val_exog_scaled, test_exog_scaled = processor.scale_exogenous_features(
            train_exog, val_exog, test_exog
        )

        # Hyperparameter tuning (optimized)
        tuner = SARIMAXTuner(logger, seasonal_period=best_period)
        param_combinations = tuner.generate_param_combinations()

        tuning_results = tuner.tune_hyperparameters(
            train_endog, train_exog_scaled,
            val_endog, val_exog_scaled,
            param_combinations,
            max_combinations=25  # OPTIMIZED
        )

        best_order = tuning_results['best_order']
        best_seasonal_order = tuning_results['best_seasonal_order']
        final_model = train_sarimax_model(
            train_endog, train_exog_scaled,
            best_order, best_seasonal_order, logger
        )

        evaluator = SARIMAXEvaluator(logger)
        all_metrics = evaluator.evaluate_model(
            final_model,
            train_endog, train_exog_scaled,
            val_endog, val_exog_scaled,
            test_endog, test_exog_scaled
        )

        full_endog = pd.concat([train_endog, val_endog])
        full_exog = np.vstack([train_exog_scaled, val_exog_scaled])

        cv_results = evaluator.time_series_cross_validation(
            full_endog, full_exog,
            best_order, best_seasonal_order,
            n_splits=3  # OPTIMIZED
        )

        save_results(
            final_model, all_metrics, cv_results, tuning_results,
            seasonality_results, processor.exog_scaler,
            output_dir='results', logger=logger
        )

        logger.info("\nSARIMAX MODEL EXPLORATION COMPLETED SUCCESSFULLY (OPTIMIZED)")

    except Exception as e:
        logger.error(f"\n\nERROR: {str(e)}", exc_info=True)
        raise


if __name__ == "__main__":
    main()

2026-03-17 22:01:06 - SARIMAX_Explorer - INFO - ================================================================================
2026-03-17 22:01:06 - SARIMAX_Explorer - INFO - ================================================================================
INFO:SARIMAX_Explorer:================================================================================
2026-03-17 22:01:06 - SARIMAX_Explorer - INFO - SARIMAX MODEL EXPLORATION INITIATED (OPTIMIZED VERSION)
2026-03-17 22:01:06 - SARIMAX_Explorer - INFO - SARIMAX MODEL EXPLORATION INITIATED (OPTIMIZED VERSION)
INFO:SARIMAX_Explorer:SARIMAX MODEL EXPLORATION INITIATED (OPTIMIZED VERSION)
2026-03-17 22:01:06 - SARIMAX_Explorer - INFO - ================================================================================
2026-03-17 22:01:06 - SARIMAX_Explorer - INFO - ================================================================================
INFO:SARIMAX_Explorer:=========================================================================

In [ ]:
"""
Holt-Winters (Triple Exponential Smoothing) Model Exploration for Sri Lankan Tourism Prediction
=================================================================================================
Author: ML Research Team
Date: December 2025
Description: Production-ready Holt-Winters implementation with trend and seasonal components,
             proper time-series validation, hyperparameter tuning, and comprehensive evaluation.
"""

import os
import logging
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Tuple, Dict, List, Optional

# Statistical and ML Libraries
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import itertools
import json
import pickle

warnings.filterwarnings('ignore')

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

def setup_logging(log_dir: str = 'logs') -> logging.Logger:
    """
    Setup comprehensive logging for model exploration.

    Args:
        log_dir: Directory to save log files

    Returns:
        Logger instance
    """
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = os.path.join(log_dir, f'holtwinters_exploration_{timestamp}.log')

    # Configure logger
    logger = logging.getLogger('HoltWinters_Explorer')
    logger.setLevel(logging.INFO)

    # File handler
    fh = logging.FileHandler(log_file)
    fh.setLevel(logging.INFO)

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)

    # Formatter
    formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)

    logger.addHandler(fh)
    logger.addHandler(ch)

    logger.info("="*80)
    logger.info("HOLT-WINTERS MODEL EXPLORATION INITIATED")
    logger.info("="*80)

    return logger


# ============================================================================
# DATA LOADING AND PREPROCESSING
# ============================================================================

class HoltWintersDataProcessor:
    """
    Holt-Winters specific data processing including seasonal analysis and outlier handling.
    Note: Holt-Winters is univariate (doesn't use exogenous features directly).
    """

    def __init__(self, logger: logging.Logger):
        self.logger = logger
        self.target_column = 'arrivals'
        self.seasonal_period = None

    def load_data(self, file_path: str) -> pd.DataFrame:
        """Load and validate the preprocessed dataset."""
        self.logger.info(f"Loading data from: {file_path}")

        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)

        self.logger.info(f"Data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")
        self.logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")

        return df

    def check_stationarity(self, series: pd.Series, name: str = "Series") -> bool:
        """
        Check stationarity using Augmented Dickey-Fuller test.

        Args:
            series: Time series to test
            name: Name of the series for logging

        Returns:
            True if stationary, False otherwise
        """
        result = adfuller(series.dropna())
        self.logger.info(f"\nStationarity Test - {name}:")
        self.logger.info(f"  ADF Statistic: {result[0]:.6f}")
        self.logger.info(f"  p-value: {result[1]:.6f}")
        self.logger.info(f"  Critical Values:")
        for key, value in result[4].items():
            self.logger.info(f"    {key}: {value:.3f}")

        is_stationary = result[1] < 0.05
        self.logger.info(f"  Stationary: {is_stationary}")

        return is_stationary

    def analyze_seasonality(self, df: pd.DataFrame, period: int = 7) -> Dict:
        """
        Analyze seasonal patterns in the data.

        Args:
            df: Input dataframe with 'date' and 'arrivals'
            period: Seasonal period to test

        Returns:
            Dictionary with seasonality statistics
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("SEASONAL DECOMPOSITION ANALYSIS")
        self.logger.info("="*80)

        ts_data = df.set_index('date')['arrivals']

        self.logger.info(f"\nAnalyzing seasonality with period = {period}")

        try:
            decomposition = seasonal_decompose(ts_data, model='additive', period=period, extrapolate_trend='freq')

            seasonal_strength = 1 - (np.var(decomposition.resid.dropna()) /
                                    np.var(decomposition.seasonal.dropna() + decomposition.resid.dropna()))

            trend_strength = 1 - (np.var(decomposition.resid.dropna()) /
                                 np.var(decomposition.trend.dropna() + decomposition.resid.dropna()))

            self.logger.info(f"\nSeasonality Analysis Results:")
            self.logger.info(f"  Seasonal Strength: {seasonal_strength:.4f}")
            self.logger.info(f"  Trend Strength: {trend_strength:.4f}")
            self.logger.info(f"  Residual Variance: {np.var(decomposition.resid.dropna()):.2f}")

            if seasonal_strength > 0.6:
                self.logger.info(f"  → Strong seasonality detected (period={period})")
            elif seasonal_strength > 0.3:
                self.logger.info(f"  → Moderate seasonality detected (period={period})")
            else:
                self.logger.info(f"  → Weak seasonality (period={period})")

            self.seasonal_period = period

            return {
                'period': period,
                'seasonal_strength': seasonal_strength,
                'trend_strength': trend_strength,
                'residual_variance': float(np.var(decomposition.resid.dropna()))
            }

        except Exception as e:
            self.logger.warning(f"Could not perform seasonal decomposition: {str(e)}")
            return {'period': period, 'seasonal_strength': 0.0}

    def detect_and_handle_outliers(self, df: pd.DataFrame,
                                   method: str = 'iqr',
                                   threshold: float = 3.0) -> pd.DataFrame:
        """
        Detect and optionally handle outliers in the target variable.

        Args:
            df: Input dataframe
            method: Method for outlier detection ('iqr' or 'zscore')
            threshold: Threshold for outlier detection

        Returns:
            DataFrame with outlier information
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("OUTLIER DETECTION AND ANALYSIS")
        self.logger.info("="*80)

        arrivals = df['arrivals'].copy()

        if method == 'iqr':
            Q1 = arrivals.quantile(0.25)
            Q3 = arrivals.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR
            outliers = (arrivals < lower_bound) | (arrivals > upper_bound)
        else:  # zscore
            z_scores = np.abs((arrivals - arrivals.mean()) / arrivals.std())
            outliers = z_scores > threshold

        n_outliers = outliers.sum()
        outlier_pct = (n_outliers / len(df)) * 100

        self.logger.info(f"\nOutlier Detection Method: {method.upper()}")
        self.logger.info(f"Threshold: {threshold}")
        self.logger.info(f"Outliers detected: {n_outliers} ({outlier_pct:.2f}%)")

        if n_outliers > 0:
            self.logger.info(f"Outlier dates: {df.loc[outliers, 'date'].tolist()[:10]}")  # Show first 10

        # Note: For Holt-Winters, we typically don't remove outliers but log them
        self.logger.info("\nNote: Outliers retained for Holt-Winters (model can handle them)")

        return df

    def prepare_time_series(self, df: pd.DataFrame) -> Tuple[pd.Series, pd.DatetimeIndex]:
        """
        Prepare time series data for Holt-Winters model.

        Args:
            df: Input dataframe

        Returns:
            Tuple of (time_series, date_index)
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("TIME SERIES PREPARATION FOR HOLT-WINTERS")
        self.logger.info("="*80)

        # Create time series with datetime index
        ts = pd.Series(df['arrivals'].values, index=df['date'])

        self.logger.info(f"\nTime series created:")
        self.logger.info(f"  Length: {len(ts)}")
        self.logger.info(f"  Start: {ts.index[0]}")
        self.logger.info(f"  End: {ts.index[-1]}")
        self.logger.info(f"  Frequency: Daily")
        self.logger.info(f"  Mean: {ts.mean():.2f}")
        self.logger.info(f"  Std: {ts.std():.2f}")
        self.logger.info(f"  Min: {ts.min():.2f}")
        self.logger.info(f"  Max: {ts.max():.2f}")

        return ts, ts.index


# ============================================================================
# DATA SPLITTING
# ============================================================================

def time_series_split(ts: pd.Series,
                     train_ratio: float = 0.75,
                     val_ratio: float = 0.15,
                     logger: logging.Logger = None) -> Tuple:
    """
    Time-series aware data splitting (75/15/15).

    Args:
        ts: Input time series
        train_ratio: Proportion for training
        val_ratio: Proportion for validation
        logger: Logger instance

    Returns:
        Tuple of (train_ts, val_ts, test_ts)
    """
    logger.info("\n" + "="*80)
    logger.info("TIME-SERIES AWARE DATA SPLITTING")
    logger.info("="*80)

    n = len(ts)
    train_size = int(n * train_ratio)
    val_size = int(n * val_ratio)

    train_ts = ts.iloc[:train_size]
    val_ts = ts.iloc[train_size:train_size + val_size]
    test_ts = ts.iloc[train_size + val_size:]

    logger.info(f"\nTotal samples: {n}")
    logger.info(f"Train: {len(train_ts)} samples ({train_ratio*100:.1f}%) - {train_ts.index[0]} to {train_ts.index[-1]}")
    logger.info(f"Validation: {len(val_ts)} samples ({val_ratio*100:.1f}%) - {val_ts.index[0]} to {val_ts.index[-1]}")
    logger.info(f"Test: {len(test_ts)} samples ({(1-train_ratio-val_ratio)*100:.1f}%) - {test_ts.index[0]} to {test_ts.index[-1]}")

    return train_ts, val_ts, test_ts


# ============================================================================
# HYPERPARAMETER TUNING
# ============================================================================

class HoltWintersTuner:
    """
    Hyperparameter tuning for Holt-Winters models using grid search.
    """

    def __init__(self, logger: logging.Logger, seasonal_period: int = 7):
        self.logger = logger
        self.seasonal_period = seasonal_period
        self.best_params = None
        self.best_model = None
        self.best_score = np.inf
        self.tuning_results = []

    def generate_param_combinations(self) -> List[Dict]:
        """
        Generate parameter combinations for Holt-Winters grid search.

        Returns:
            List of parameter dictionaries
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("GENERATING HOLT-WINTERS PARAMETER COMBINATIONS")
        self.logger.info("="*80)

        # Holt-Winters parameters
        trend_options = ['add', 'mul', None]
        seasonal_options = ['add', 'mul', None]
        damped_trend_options = [True, False]

        # Generate all combinations
        param_list = []
        for trend in trend_options:
            for seasonal in seasonal_options:
                for damped in damped_trend_options:
                    # Skip damped trend if no trend
                    if trend is None and damped:
                        continue
                    # Skip damped trend with multiplicative trend (not supported)
                    if trend == 'mul' and damped:
                        continue

                    params = {
                        'trend': trend,
                        'seasonal': seasonal,
                        'damped_trend': damped,
                        'seasonal_periods': self.seasonal_period if seasonal is not None else None
                    }
                    param_list.append(params)

        self.logger.info(f"\nGenerated {len(param_list)} parameter combinations")
        self.logger.info(f"Seasonal period: {self.seasonal_period}")
        self.logger.info("\nParameter options:")
        self.logger.info(f"  Trend: {trend_options}")
        self.logger.info(f"  Seasonal: {seasonal_options}")
        self.logger.info(f"  Damped: {damped_trend_options}")

        return param_list

    def tune_hyperparameters(self,
                           train_ts: pd.Series,
                           val_ts: pd.Series,
                           param_combinations: List[Dict]) -> Dict:
        """
        Perform grid search for hyperparameter tuning.

        Args:
            train_ts: Training time series
            val_ts: Validation time series
            param_combinations: List of parameter dictionaries to test

        Returns:
            Dictionary with best parameters
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("HYPERPARAMETER TUNING - GRID SEARCH")
        self.logger.info("="*80)

        self.logger.info(f"\nTesting {len(param_combinations)} parameter combinations...")

        for idx, params in enumerate(param_combinations, 1):
            try:
                # Fit model on training data
                model = ExponentialSmoothing(
                    train_ts,
                    trend=params['trend'],
                    seasonal=params['seasonal'],
                    seasonal_periods=params['seasonal_periods'],
                    damped_trend=params['damped_trend']
                )

                fitted_model = model.fit(optimized=True, use_brute=True)

                # Forecast on validation set
                val_predictions = fitted_model.forecast(steps=len(val_ts))
                val_mse = mean_squared_error(val_ts, val_predictions)
                val_rmse = np.sqrt(val_mse)

                # Record results
                result = {
                    'trend': params['trend'],
                    'seasonal': params['seasonal'],
                    'damped_trend': params['damped_trend'],
                    'seasonal_periods': params['seasonal_periods'],
                    'aic': fitted_model.aic,
                    'bic': fitted_model.bic,
                    'val_mse': val_mse,
                    'val_rmse': val_rmse,
                    'smoothing_level': fitted_model.params['smoothing_level'],
                    'smoothing_trend': fitted_model.params.get('smoothing_trend', None),
                    'smoothing_seasonal': fitted_model.params.get('smoothing_seasonal', None)
                }
                self.tuning_results.append(result)

                # Update best model based on validation RMSE
                if val_rmse < self.best_score:
                    self.best_score = val_rmse
                    self.best_params = params
                    self.best_model = fitted_model
                    self.logger.info(f"  [{idx}/{len(param_combinations)}] New best model found!")
                    self.logger.info(f"    Trend: {params['trend']}, Seasonal: {params['seasonal']}, Damped: {params['damped_trend']}")
                    self.logger.info(f"    Val RMSE: {val_rmse:.2f}, AIC: {fitted_model.aic:.2f}")

                if idx % 3 == 0:
                    self.logger.info(f"  Progress: {idx}/{len(param_combinations)} combinations tested")

            except Exception as e:
                self.logger.warning(f"  Failed for params {params}: {str(e)[:100]}")
                continue

        self.logger.info("\n" + "-"*80)
        self.logger.info("HYPERPARAMETER TUNING COMPLETED")
        self.logger.info(f"Best parameters: {self.best_params}")
        self.logger.info(f"Best validation RMSE: {self.best_score:.2f}")
        self.logger.info("-"*80)

        return {
            'best_params': self.best_params,
            'best_val_rmse': self.best_score,
            'all_results': self.tuning_results
        }


# ============================================================================
# MODEL TRAINING
# ============================================================================

def train_holtwinters_model(train_ts: pd.Series,
                           params: Dict,
                           logger: logging.Logger) -> ExponentialSmoothing:
    """
    Train Holt-Winters model with best parameters.

    Args:
        train_ts: Training time series
        params: Model parameters
        logger: Logger instance

    Returns:
        Fitted ExponentialSmoothing model
    """
    logger.info("\n" + "="*80)
    logger.info("MODEL TRAINING WITH BEST PARAMETERS")
    logger.info("="*80)

    logger.info(f"\nTraining Holt-Winters model...")
    logger.info(f"Training samples: {len(train_ts)}")
    logger.info(f"Parameters:")
    for key, value in params.items():
        logger.info(f"  {key}: {value}")

    model = ExponentialSmoothing(
        train_ts,
        trend=params['trend'],
        seasonal=params['seasonal'],
        seasonal_periods=params['seasonal_periods'],
        damped_trend=params['damped_trend']
    )

    fitted_model = model.fit(optimized=True, use_brute=True)

    logger.info("\nModel training completed successfully!")
    logger.info(f"AIC: {fitted_model.aic:.2f}")
    logger.info(f"BIC: {fitted_model.bic:.2f}")
    logger.info(f"\nOptimized Parameters:")
    logger.info(f"  Alpha (level): {fitted_model.params['smoothing_level']:.4f}")
    if fitted_model.params.get('smoothing_trend') is not None:
        logger.info(f"  Beta (trend): {fitted_model.params['smoothing_trend']:.4f}")
    if fitted_model.params.get('smoothing_seasonal') is not None:
        logger.info(f"  Gamma (seasonal): {fitted_model.params['smoothing_seasonal']:.4f}")
    if fitted_model.params.get('damping_trend') is not None:
        logger.info(f"  Phi (damping): {fitted_model.params['damping_trend']:.4f}")

    return fitted_model


# ============================================================================
# MODEL EVALUATION
# ============================================================================

class HoltWintersEvaluator:
    """
    Comprehensive evaluation of Holt-Winters models.
    """

    def __init__(self, logger: logging.Logger):
        self.logger = logger
        self.metrics = {}

    def calculate_metrics(self, y_true: np.ndarray, y_pred: np.ndarray,
                         dataset_name: str = "Dataset") -> Dict:
        """
        Calculate comprehensive evaluation metrics.

        Args:
            y_true: True values
            y_pred: Predicted values
            dataset_name: Name of the dataset

        Returns:
            Dictionary of metrics
        """
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mape = mean_absolute_percentage_error(y_true, y_pred) * 100
        r2 = r2_score(y_true, y_pred)
        mae = np.mean(np.abs(y_true - y_pred))

        metrics = {
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'MAPE': mape,
            'R2': r2
        }

        self.logger.info(f"\n{dataset_name} Metrics:")
        self.logger.info(f"  MSE:  {mse:.2f}")
        self.logger.info(f"  RMSE: {rmse:.2f}")
        self.logger.info(f"  MAE:  {mae:.2f}")
        self.logger.info(f"  MAPE: {mape:.2f}%")
        self.logger.info(f"  R²:   {r2:.4f}")

        return metrics

    def time_series_cross_validation(self,
                                    ts: pd.Series,
                                    params: Dict,
                                    n_splits: int = 5) -> Dict:
        """
        Perform time series cross-validation.

        Args:
            ts: Time series
            params: Model parameters
            n_splits: Number of cross-validation splits

        Returns:
            Dictionary with CV metrics
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("TIME SERIES CROSS-VALIDATION")
        self.logger.info("="*80)

        self.logger.info(f"\nPerforming {n_splits}-fold time series cross-validation...")

        n = len(ts)
        fold_size = n // (n_splits + 1)

        cv_scores = {
            'rmse': [],
            'mse': [],
            'mae': [],
            'mape': [],
            'r2': []
        }

        for i in range(n_splits):
            train_end = fold_size * (i + 2)
            test_start = train_end
            test_end = min(train_end + fold_size, n)

            if test_end <= test_start:
                break

            train_ts = ts.iloc[:train_end]
            test_ts = ts.iloc[test_start:test_end]

            try:
                # Train model
                model = ExponentialSmoothing(
                    train_ts,
                    trend=params['trend'],
                    seasonal=params['seasonal'],
                    seasonal_periods=params['seasonal_periods'],
                    damped_trend=params['damped_trend']
                )
                fitted = model.fit(optimized=True, use_brute=True)

                # Forecast
                predictions = fitted.forecast(steps=len(test_ts))

                # Calculate metrics
                mse = mean_squared_error(test_ts, predictions)
                rmse = np.sqrt(mse)
                mae = np.mean(np.abs(test_ts - predictions))
                mape = mean_absolute_percentage_error(test_ts, predictions) * 100
                r2 = r2_score(test_ts, predictions)

                cv_scores['mse'].append(mse)
                cv_scores['rmse'].append(rmse)
                cv_scores['mae'].append(mae)
                cv_scores['mape'].append(mape)
                cv_scores['r2'].append(r2)

                self.logger.info(f"\nFold {i+1}/{n_splits}:")
                self.logger.info(f"  Train size: {len(train_ts)}, Test size: {len(test_ts)}")
                self.logger.info(f"  RMSE: {rmse:.2f}, MAPE: {mape:.2f}%, R²: {r2:.4f}")

            except Exception as e:
                self.logger.warning(f"Fold {i+1} failed: {str(e)}")
                continue

        # Calculate average scores
        cv_results = {
            'mean_rmse': np.mean(cv_scores['rmse']),
            'std_rmse': np.std(cv_scores['rmse']),
            'mean_mse': np.mean(cv_scores['mse']),
            'std_mse': np.std(cv_scores['mse']),
            'mean_mae': np.mean(cv_scores['mae']),
            'std_mae': np.std(cv_scores['mae']),
            'mean_mape': np.mean(cv_scores['mape']),
            'std_mape': np.std(cv_scores['mape']),
            'mean_r2': np.mean(cv_scores['r2']),
            'std_r2': np.std(cv_scores['r2']),
            'n_folds_completed': len(cv_scores['rmse'])
        }

        self.logger.info("\n" + "-"*80)
        self.logger.info("CROSS-VALIDATION RESULTS:")
        self.logger.info(f"  Completed folds: {cv_results['n_folds_completed']}/{n_splits}")
        self.logger.info(f"  Mean RMSE: {cv_results['mean_rmse']:.2f} ± {cv_results['std_rmse']:.2f}")
        self.logger.info(f"  Mean MAE:  {cv_results['mean_mae']:.2f} ± {cv_results['std_mae']:.2f}")
        self.logger.info(f"  Mean MAPE: {cv_results['mean_mape']:.2f}% ± {cv_results['std_mape']:.2f}%")
        self.logger.info(f"  Mean R²:   {cv_results['mean_r2']:.4f} ± {cv_results['std_r2']:.4f}")
        self.logger.info("-"*80)

        return cv_results

    def evaluate_model(self,
                      model: ExponentialSmoothing,
                      train_ts: pd.Series,
                      val_ts: pd.Series,
                      test_ts: pd.Series) -> Dict:
        """
        Comprehensive model evaluation on all datasets.

        Args:
            model: Fitted ExponentialSmoothing model
            train_ts: Training time series
            val_ts: Validation time series
            test_ts: Test time series

        Returns:
            Dictionary with all evaluation metrics
        """
        self.logger.info("\n" + "="*80)
        self.logger.info("COMPREHENSIVE MODEL EVALUATION")
        self.logger.info("="*80)

        # Training set evaluation (fitted values)
        train_pred = model.fittedvalues
        train_metrics = self.calculate_metrics(train_ts, train_pred, "Training Set")

        # Validation set evaluation
        val_pred = model.forecast(steps=len(val_ts))
        val_metrics = self.calculate_metrics(val_ts, val_pred, "Validation Set")

        # Test set evaluation
        # Need to refit including validation data for proper test forecast
        combined_ts = pd.concat([train_ts, val_ts])
        model_full = ExponentialSmoothing(
            combined_ts,
            trend=model.model.trend,
            seasonal=model.model.seasonal,
            seasonal_periods=model.model.seasonal_periods,
            damped_trend=model.model.damped_trend
        )
        fitted_full = model_full.fit(optimized=True, use_brute=True)
        test_pred = fitted_full.forecast(steps=len(test_ts))
        test_metrics = self.calculate_metrics(test_ts, test_pred, "Test Set")

        all_metrics = {
            'train': train_metrics,
            'validation': val_metrics,
            'test': test_metrics
        }

        self.metrics = all_metrics

        return all_metrics


# ============================================================================
# RESULTS SAVING
# ============================================================================

def save_results(model: ExponentialSmoothing,
                metrics: Dict,
                cv_results: Dict,
                tuning_results: Dict,
                seasonality_analysis: Dict,
                output_dir: str = 'results',
                logger: logging.Logger = None):
    """
    Save model, metrics, and results to disk.

    Args:
        model: Fitted ExponentialSmoothing model
        metrics: Evaluation metrics
        cv_results: Cross-validation results
        tuning_results: Hyperparameter tuning results
        seasonality_analysis: Seasonality analysis results
        output_dir: Directory to save results
        logger: Logger instance
    """
    logger.info("\n" + "="*80)
    logger.info("SAVING RESULTS")
    logger.info("="*80)

    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Save model
    model_path = os.path.join(output_dir, f'holtwinters_model_{timestamp}.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    logger.info(f"\nModel saved: {model_path}")

    # Save metrics
    metrics_path = os.path.join(output_dir, f'metrics_{timestamp}.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)
    logger.info(f"Metrics saved: {metrics_path}")

    # Save CV results
    cv_path = os.path.join(output_dir, f'cv_results_{timestamp}.json')
    with open(cv_path, 'w') as f:
        json.dump(cv_results, f, indent=4)
    logger.info(f"CV results saved: {cv_path}")

    # Save seasonality analysis
    seasonality_path = os.path.join(output_dir, f'seasonality_analysis_{timestamp}.json')
    with open(seasonality_path, 'w') as f:
        json.dump(seasonality_analysis, f, indent=4)
    logger.info(f"Seasonality analysis saved: {seasonality_path}")

    # Save tuning results
    tuning_path = os.path.join(output_dir, f'tuning_results_{timestamp}.json')
    # Make serializable
    tuning_serializable = {
        'best_params': tuning_results['best_params'],
        'best_val_rmse': float(tuning_results['best_val_rmse']),
        'all_results': [
            {
                'trend': r['trend'],
                'seasonal': r['seasonal'],
                'damped_trend': r['damped_trend'],
                'seasonal_periods': r['seasonal_periods'],
                'aic': float(r['aic']),
                'bic': float(r['bic']),
                'val_mse': float(r['val_mse']),
                'val_rmse': float(r['val_rmse']),
                'smoothing_level': float(r['smoothing_level']),
                'smoothing_trend': float(r['smoothing_trend']) if r['smoothing_trend'] is not None else None,
                'smoothing_seasonal': float(r['smoothing_seasonal']) if r['smoothing_seasonal'] is not None else None
            }
            for r in tuning_results['all_results']
        ]
    }
    with open(tuning_path, 'w') as f:
        json.dump(tuning_serializable, f, indent=4)
    logger.info(f"Tuning results saved: {tuning_path}")

    # Save summary report
    summary_path = os.path.join(output_dir, f'model_summary_{timestamp}.txt')
    with open(summary_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("HOLT-WINTERS MODEL EXPLORATION - SUMMARY REPORT\n")
        f.write("="*80 + "\n\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("BEST MODEL PARAMETERS:\n")
        f.write("-"*80 + "\n")
        for key, value in tuning_results['best_params'].items():
            f.write(f"  {key}: {value}\n")
        f.write("\n" + "-"*80 + "\n")
        f.write("\nSEASONALITY ANALYSIS:\n")
        f.write("-"*80 + "\n")
        for key, value in seasonality_analysis.items():
            if isinstance(value, dict):
                f.write(f"  {key}:\n")
                for k, v in value.items():
                    f.write(f"    {k}: {v}\n")
            else:
                f.write(f"  {key}: {value}\n")
        f.write("\n" + "-"*80 + "\n")
        f.write("\nEVALUATION METRICS:\n")
        f.write("-"*80 + "\n")
        for dataset, dataset_metrics in metrics.items():
            f.write(f"\n{dataset.upper()}:\n")
            for metric, value in dataset_metrics.items():
                f.write(f"  {metric}: {value:.4f}\n")
        f.write("\n" + "-"*80 + "\n")
        f.write("\nCROSS-VALIDATION RESULTS:\n")
        f.write("-"*80 + "\n")
        for metric, value in cv_results.items():
            f.write(f"  {metric}: {value:.4f}\n")
    logger.info(f"Summary report saved: {summary_path}")

    logger.info("\n" + "="*80)
    logger.info("ALL RESULTS SAVED SUCCESSFULLY")
    logger.info("="*80)


# ============================================================================
# MAIN EXECUTION PIPELINE
# ============================================================================

def main():
    """
    Main execution pipeline for Holt-Winters model exploration.
    """
    # Setup logging
    logger = setup_logging()

    try:
        # 1. Initialize data processor
        logger.info("\nInitializing Holt-Winters data processor...")
        processor = HoltWintersDataProcessor(logger)

        # 2. Load data
        df = processor.load_data('preprocessed-dataset.csv')

        # 3. Analyze seasonality (test different periods)
        seasonality_results = {}
        for period in [7, 30]:  # Weekly and monthly
            season_stats = processor.analyze_seasonality(df, period=period)
            seasonality_results[f'period_{period}'] = season_stats

        # Choose seasonal period with strongest seasonality
        best_period = 7  # Default to weekly
        best_strength = 0
        for period_key, stats in seasonality_results.items():
            if stats.get('seasonal_strength', 0) > best_strength:
                best_strength = stats['seasonal_strength']
                best_period = stats['period']

        processor.seasonal_period = best_period
        logger.info(f"\nSelected seasonal period: {best_period} (strength: {best_strength:.4f})")

        # 4. Check stationarity
        processor.check_stationarity(df['arrivals'], "Tourist Arrivals")

        # 5. Detect outliers
        df = processor.detect_and_handle_outliers(df, method='iqr', threshold=3.0)

        # 6. Prepare time series
        ts, date_index = processor.prepare_time_series(df)

        # 7. Time-series split
        train_ts, val_ts, test_ts = time_series_split(
            ts, train_ratio=0.75, val_ratio=0.15, logger=logger
        )

        # 8. Hyperparameter tuning
        tuner = HoltWintersTuner(logger, seasonal_period=best_period)
        param_combinations = tuner.generate_param_combinations()

        tuning_results = tuner.tune_hyperparameters(
            train_ts, val_ts, param_combinations
        )

        # 9. Train final model with best parameters
        best_params = tuning_results['best_params']
        final_model = train_holtwinters_model(train_ts, best_params, logger)

        # 10. Evaluate model
        evaluator = HoltWintersEvaluator(logger)
        all_metrics = evaluator.evaluate_model(
            final_model, train_ts, val_ts, test_ts
        )

        # 11. Time series cross-validation
        full_ts = pd.concat([train_ts, val_ts])
        cv_results = evaluator.time_series_cross_validation(
            full_ts, best_params, n_splits=5
        )

        # 12. Save all results
        save_results(
            final_model, all_metrics, cv_results, tuning_results,
            seasonality_results, output_dir='results', logger=logger
        )

        logger.info("\n" + "="*80)
        logger.info("HOLT-WINTERS MODEL EXPLORATION COMPLETED SUCCESSFULLY")
        logger.info("="*80)

    except Exception as e:
        logger.error(f"\n\nERROR: {str(e)}", exc_info=True)
        raise


if __name__ == "__main__":
    main()

2026-03-18 04:10:35 - HoltWinters_Explorer - INFO - ================================================================================
INFO:HoltWinters_Explorer:================================================================================
2026-03-18 04:10:35 - HoltWinters_Explorer - INFO - HOLT-WINTERS MODEL EXPLORATION INITIATED
INFO:HoltWinters_Explorer:HOLT-WINTERS MODEL EXPLORATION INITIATED
2026-03-18 04:10:35 - HoltWinters_Explorer - INFO - ================================================================================
INFO:HoltWinters_Explorer:================================================================================
2026-03-18 04:10:35 - HoltWinters_Explorer - INFO - 
Initializing Holt-Winters data processor...
INFO:HoltWinters_Explorer:
Initializing Holt-Winters data processor...
2026-03-18 04:10:35 - HoltWinters_Explorer - INFO - Loading data from: preprocessed-dataset.csv
INFO:HoltWinters_Explorer:Loading data from: preprocessed-dataset.csv
2026-03-18 04:10:35 - Holt

In [ ]:
!pip install prophet

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import itertools
import logging
import json
import warnings
from datetime import datetime
import pickle

warnings.filterwarnings('ignore')

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

def setup_logging():
    """Setup logging configuration for the model training process"""
    log_filename = f'prophet_model_exploration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

logger = setup_logging()

# ============================================================================
# 1. MODEL-SPECIFIC PREPROCESSING
# ============================================================================

def prepare_prophet_data(df, target_col='arrivals'):
    """
    Prepare data for Prophet model format

    Prophet requires:
    - 'ds' column: date column
    - 'y' column: target variable
    - Additional regressors as separate columns

    Args:
        df: pandas DataFrame with date and features
        target_col: name of target column

    Returns:
        DataFrame in Prophet format
    """
    logger.info("Starting Prophet-specific data preprocessing...")

    # Create a copy to avoid modifying original
    prophet_df = df.copy()

    # Rename columns for Prophet
    prophet_df = prophet_df.rename(columns={
        'date': 'ds',
        target_col: 'y'
    })

    # Ensure 'ds' is datetime
    prophet_df['ds'] = pd.to_datetime(prophet_df['ds'])

    # Sort by date
    prophet_df = prophet_df.sort_values('ds').reset_index(drop=True)

    # Log data shape and date range
    logger.info(f"Prophet data shape: {prophet_df.shape}")
    logger.info(f"Date range: {prophet_df['ds'].min()} to {prophet_df['ds'].max()}")
    logger.info(f"Target variable stats - Mean: {prophet_df['y'].mean():.2f}, Std: {prophet_df['y'].std():.2f}")

    return prophet_df

def get_regressor_columns(df):
    """Extract regressor column names (exclude 'ds' and 'y')"""
    regressors = [col for col in df.columns if col not in ['ds', 'y']]
    logger.info(f"Found {len(regressors)} regressors: {regressors}")
    return regressors

# ============================================================================
# 2. TIME-SERIES AWARE DATA SPLITTING
# ============================================================================

def time_series_split(df, train_ratio=0.75, val_ratio=0.15, test_ratio=0.10):
    """
    Split time series data maintaining temporal order

    Args:
        df: DataFrame with time series data
        train_ratio: proportion for training (default 0.75)
        val_ratio: proportion for validation (default 0.15)
        test_ratio: proportion for testing (default 0.10)

    Returns:
        train_df, val_df, test_df
    """
    logger.info("="*80)
    logger.info("SPLITTING DATA - Time Series Aware Split")
    logger.info("="*80)

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"

    n = len(df)
    train_size = int(n * train_ratio)
    val_size = int(n * val_ratio)

    train_df = df.iloc[:train_size].copy()
    val_df = df.iloc[train_size:train_size + val_size].copy()
    test_df = df.iloc[train_size + val_size:].copy()

    logger.info(f"Total samples: {n}")
    logger.info(f"Train set: {len(train_df)} samples ({train_ratio*100:.1f}%) | {train_df['ds'].min()} to {train_df['ds'].max()}")
    logger.info(f"Validation set: {len(val_df)} samples ({val_ratio*100:.1f}%) | {val_df['ds'].min()} to {val_df['ds'].max()}")
    logger.info(f"Test set: {len(test_df)} samples ({test_ratio*100:.1f}%) | {test_df['ds'].min()} to {test_df['ds'].max()}")

    return train_df, val_df, test_df

# ============================================================================
# 3. HYPERPARAMETER TUNING
# ============================================================================

def generate_param_grid():
    """
    Generate hyperparameter grid for Prophet model

    Key hyperparameters:
    - changepoint_prior_scale: Flexibility of trend (higher = more flexible)
    - seasonality_prior_scale: Flexibility of seasonality
    - holidays_prior_scale: Flexibility of holiday effects
    - seasonality_mode: 'additive' or 'multiplicative'
    - changepoint_range: Proportion of history for trend changepoints

    Returns:
        List of parameter dictionaries
    """
    param_grid = {
        'changepoint_prior_scale': [0.001, 0.01, 0.1, 0.5],
        'seasonality_prior_scale': [0.01, 0.1, 1.0, 10.0],
        'holidays_prior_scale': [0.01, 0.1, 1.0, 10.0],
        'seasonality_mode': ['additive', 'multiplicative'],
        'changepoint_range': [0.8, 0.9]
    }

    # Generate all combinations
    keys = param_grid.keys()
    values = param_grid.values()
    param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    logger.info(f"Generated {len(param_combinations)} parameter combinations for tuning")

    return param_combinations

def evaluate_params(params, train_df, val_df, regressors):
    """
    Evaluate a single parameter configuration

    Args:
        params: dictionary of hyperparameters
        train_df: training data
        val_df: validation data
        regressors: list of regressor column names

    Returns:
        params with validation MAPE score
    """
    try:
        # Initialize model with parameters
        model = Prophet(
            changepoint_prior_scale=params['changepoint_prior_scale'],
            seasonality_prior_scale=params['seasonality_prior_scale'],
            holidays_prior_scale=params['holidays_prior_scale'],
            seasonality_mode=params['seasonality_mode'],
            changepoint_range=params['changepoint_range'],
            daily_seasonality=False,
            weekly_seasonality=True,
            yearly_seasonality=True,
            interval_width=0.95
        )

        # Add regressors
        for regressor in regressors:
            model.add_regressor(regressor)

        # Fit model
        model.fit(train_df, verbose=False)

        # Predict on validation set
        forecast = model.predict(val_df[['ds'] + regressors])

        # Calculate MAPE
        y_true = val_df['y'].values
        y_pred = forecast['yhat'].values
        mape = mean_absolute_percentage_error(y_true, y_pred)

        params['val_mape'] = mape
        return params

    except Exception as e:
        logger.warning(f"Error evaluating params {params}: {str(e)}")
        params['val_mape'] = np.inf
        return params

def hyperparameter_tuning(train_df, val_df, regressors, max_evals=None):
    """
    Perform hyperparameter tuning using grid search

    Args:
        train_df: training data
        val_df: validation data
        regressors: list of regressor names
        max_evals: maximum number of evaluations (None = all combinations)

    Returns:
        best_params: dictionary of best hyperparameters
    """
    logger.info("="*80)
    logger.info("HYPERPARAMETER TUNING")
    logger.info("="*80)

    param_combinations = generate_param_grid()

    if max_evals and max_evals < len(param_combinations):
        # Random sampling for faster tuning
        np.random.seed(42)
        param_combinations = np.random.choice(param_combinations, max_evals, replace=False).tolist()
        logger.info(f"Sampling {max_evals} random combinations for faster tuning")

    results = []
    total = len(param_combinations)

    logger.info(f"Evaluating {total} parameter combinations...")

    for idx, params in enumerate(param_combinations, 1):
        if idx % 10 == 0 or idx == 1:
            logger.info(f"Progress: {idx}/{total} ({idx/total*100:.1f}%)")

        result = evaluate_params(params, train_df, val_df, regressors)
        results.append(result)

    # Sort by validation MAPE
    results.sort(key=lambda x: x['val_mape'])

    best_params = results[0]
    logger.info(f"\nBest Parameters Found:")
    logger.info(f"  - changepoint_prior_scale: {best_params['changepoint_prior_scale']}")
    logger.info(f"  - seasonality_prior_scale: {best_params['seasonality_prior_scale']}")
    logger.info(f"  - holidays_prior_scale: {best_params['holidays_prior_scale']}")
    logger.info(f"  - seasonality_mode: {best_params['seasonality_mode']}")
    logger.info(f"  - changepoint_range: {best_params['changepoint_range']}")
    logger.info(f"  - Validation MAPE: {best_params['val_mape']:.4f}")

    # Log top 5 configurations
    logger.info("\nTop 5 Configurations:")
    for i, result in enumerate(results[:5], 1):
        logger.info(f"  {i}. MAPE: {result['val_mape']:.4f} | Params: {result}")

    return best_params

# ============================================================================
# 4. MODEL TRAINING WITH BEST PARAMETERS
# ============================================================================

def train_final_model(train_df, best_params, regressors):
    """
    Train final Prophet model with best hyperparameters

    Args:
        train_df: training data
        best_params: dictionary of best hyperparameters
        regressors: list of regressor names

    Returns:
        trained Prophet model
    """
    logger.info("="*80)
    logger.info("TRAINING FINAL MODEL WITH BEST PARAMETERS")
    logger.info("="*80)

    # Initialize model
    model = Prophet(
        changepoint_prior_scale=best_params['changepoint_prior_scale'],
        seasonality_prior_scale=best_params['seasonality_prior_scale'],
        holidays_prior_scale=best_params['holidays_prior_scale'],
        seasonality_mode=best_params['seasonality_mode'],
        changepoint_range=best_params['changepoint_range'],
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        interval_width=0.95
    )

    # Add all regressors
    for regressor in regressors:
        model.add_regressor(regressor, standardize='auto')
        logger.info(f"Added regressor: {regressor}")

    # Fit model
    logger.info("Fitting model on training data...")
    model.fit(train_df)
    logger.info("Model training completed successfully!")

    return model

# ============================================================================
# 5. MODEL EVALUATION
# ============================================================================

def calculate_metrics(y_true, y_pred, dataset_name=''):
    """
    Calculate comprehensive evaluation metrics

    Args:
        y_true: true values
        y_pred: predicted values
        dataset_name: name of dataset (train/val/test)

    Returns:
        dictionary of metrics
    """
    metrics = {
        'dataset': dataset_name,
        'r2_score': r2_score(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mse': mean_squared_error(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'mape': mean_absolute_percentage_error(y_true, y_pred)
    }

    return metrics

def evaluate_model(model, train_df, val_df, test_df, regressors):
    """
    Comprehensive model evaluation on all datasets

    Args:
        model: trained Prophet model
        train_df, val_df, test_df: data splits
        regressors: list of regressor names

    Returns:
        dictionary with all metrics and predictions
    """
    logger.info("="*80)
    logger.info("MODEL EVALUATION")
    logger.info("="*80)

    all_metrics = {}
    predictions = {}

    # Evaluate on each dataset
    for df, name in [(train_df, 'train'), (val_df, 'val'), (test_df, 'test')]:
        logger.info(f"\nEvaluating on {name.upper()} set...")

        # Make predictions
        forecast = model.predict(df[['ds'] + regressors])

        # Extract predictions
        y_true = df['y'].values
        y_pred = forecast['yhat'].values

        # Calculate metrics
        metrics = calculate_metrics(y_true, y_pred, name)
        all_metrics[name] = metrics

        # Store predictions
        predictions[name] = {
            'y_true': y_true,
            'y_pred': y_pred,
            'dates': df['ds'].values,
            'lower_bound': forecast['yhat_lower'].values,
            'upper_bound': forecast['yhat_upper'].values
        }

        # Log metrics
        logger.info(f"{name.upper()} Metrics:")
        logger.info(f"  R2 Score: {metrics['r2_score']:.4f}")
        logger.info(f"  RMSE: {metrics['rmse']:.2f}")
        logger.info(f"  MSE: {metrics['mse']:.2f}")
        logger.info(f"  MAE: {metrics['mae']:.2f}")
        logger.info(f"  MAPE: {metrics['mape']:.4f} ({metrics['mape']*100:.2f}%)")

    return all_metrics, predictions

def prophet_cross_validation(model, train_df, horizon='90 days', initial='730 days', period='30 days'):
    """
    Perform Prophet time series cross-validation

    Args:
        model: trained Prophet model
        train_df: training data
        horizon: forecast horizon
        initial: initial training period
        period: spacing between cutoff dates

    Returns:
        cross-validation metrics
    """
    logger.info("="*80)
    logger.info("TIME SERIES CROSS-VALIDATION")
    logger.info("="*80)
    logger.info(f"Parameters: horizon={horizon}, initial={initial}, period={period}")

    try:
        # Perform cross-validation
        df_cv = cross_validation(
            model,
            initial=initial,
            period=period,
            horizon=horizon,
            parallel="processes"
        )

        # Calculate performance metrics
        df_p = performance_metrics(df_cv)

        logger.info("\nCross-Validation Results:")
        logger.info(f"  Mean MAPE: {df_p['mape'].mean():.4f}")
        logger.info(f"  Mean RMSE: {df_p['rmse'].mean():.2f}")
        logger.info(f"  Mean MAE: {df_p['mae'].mean():.2f}")

        return df_cv, df_p

    except Exception as e:
        logger.error(f"Cross-validation failed: {str(e)}")
        return None, None

# ============================================================================
# 6. SAVE RESULTS
# ============================================================================

def save_results(model, best_params, all_metrics, predictions, cv_results=None):
    """
    Save model, parameters, metrics, and predictions

    Args:
        model: trained Prophet model
        best_params: best hyperparameters
        all_metrics: evaluation metrics
        predictions: predictions for all datasets
        cv_results: cross-validation results (optional)
    """
    logger.info("="*80)
    logger.info("SAVING RESULTS")
    logger.info("="*80)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save model
    model_filename = f'prophet_model_{timestamp}.pkl'
    with open(model_filename, 'wb') as f:
        pickle.dump(model, f)
    logger.info(f"Model saved: {model_filename}")

    # Save parameters
    params_filename = f'prophet_best_params_{timestamp}.json'
    with open(params_filename, 'w') as f:
        json.dump(best_params, f, indent=4)
    logger.info(f"Best parameters saved: {params_filename}")

    # Save metrics
    metrics_filename = f'prophet_metrics_{timestamp}.json'
    with open(metrics_filename, 'w') as f:
        json.dump(all_metrics, f, indent=4)
    logger.info(f"Metrics saved: {metrics_filename}")

    # Save predictions
    for dataset_name, pred_data in predictions.items():
        pred_df = pd.DataFrame({
            'date': pred_data['dates'],
            'y_true': pred_data['y_true'],
            'y_pred': pred_data['y_pred'],
            'lower_bound': pred_data['lower_bound'],
            'upper_bound': pred_data['upper_bound']
        })
        pred_filename = f'prophet_predictions_{dataset_name}_{timestamp}.csv'
        pred_df.to_csv(pred_filename, index=False)
        logger.info(f"{dataset_name.capitalize()} predictions saved: {pred_filename}")

    # Save cross-validation results if available
    if cv_results is not None:
        cv_filename = f'prophet_cv_results_{timestamp}.csv'
        cv_results.to_csv(cv_filename, index=False)
        logger.info(f"Cross-validation results saved: {cv_filename}")

    logger.info("All results saved successfully!")

# ============================================================================
# MAIN EXECUTION PIPELINE
# ============================================================================

def main(data_path, target_col='arrivals', max_tuning_evals=50):
    """
    Main execution pipeline for Prophet model

    Args:
        data_path: path to processed data CSV
        target_col: name of target column
        max_tuning_evals: maximum hyperparameter combinations to evaluate
    """
    logger.info("="*80)
    logger.info("PROPHET MODEL EXPLORATION PIPELINE")
    logger.info("="*80)
    logger.info(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    try:
        # Load data
        logger.info(f"\nLoading data from: {data_path}")
        df = pd.read_csv(data_path)
        logger.info(f"Data loaded successfully! Shape: {df.shape}")

        # 1. Prophet-specific preprocessing
        prophet_df = prepare_prophet_data(df, target_col)
        regressors = get_regressor_columns(prophet_df)

        # 2. Time series split
        train_df, val_df, test_df = time_series_split(prophet_df)

        # 3. Hyperparameter tuning
        best_params = hyperparameter_tuning(train_df, val_df, regressors, max_tuning_evals)

        # 4. Train final model
        model = train_final_model(train_df, best_params, regressors)

        # 5. Model evaluation
        all_metrics, predictions = evaluate_model(model, train_df, val_df, test_df, regressors)

        # 5b. Cross-validation (optional but recommended)
        cv_results, cv_metrics = prophet_cross_validation(model, train_df)

        # 6. Save results
        save_results(model, best_params, all_metrics, predictions, cv_results)

        logger.info("="*80)
        logger.info("PIPELINE COMPLETED SUCCESSFULLY!")
        logger.info("="*80)
        logger.info(f"End Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

        return model, best_params, all_metrics

    except Exception as e:
        logger.error(f"Pipeline failed with error: {str(e)}", exc_info=True)
        raise

# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    # Configuration
    DATA_PATH = 'preprocessed-dataset.csv'  # Update with your actual file path
    TARGET_COLUMN = 'arrivals'
    MAX_TUNING_EVALUATIONS = 50  # Reduce for faster exploration, increase for thorough search

    # Run pipeline
    model, best_params, metrics = main(
        data_path=DATA_PATH,
        target_col=TARGET_COLUMN,
        max_tuning_evals=MAX_TUNING_EVALUATIONS
    )

    # Print summary
    print("\n" + "="*80)
    print("FINAL SUMMARY")
    print("="*80)
    print("\nModel Performance:")
    for dataset in ['train', 'val', 'test']:
        print(f"\n{dataset.upper()} Set:")
        print(f"  R2 Score: {metrics[dataset]['r2_score']:.4f}")
        print(f"  RMSE: {metrics[dataset]['rmse']:.2f}")
        print(f"  MAPE: {metrics[dataset]['mape']*100:.2f}%")
